In [1]:
%cd /home/our_storage/liam_sbhoo/dl-lab-final-project

/mnt/disks/our_storage/liam_sbhoo/dl-lab-final-project


In [2]:
import os
import pprint

from omegaconf import OmegaConf
import numpy as np
import torch
from torch.utils.data import DataLoader
from lavis.models import load_preprocess

from models.localized_blip2_qformer import LocalizedBlip2QFormer
from datasets.refcoco_dataset import RefCocoDataset
from datasets.visual_genome_dataset import VGDataset
from retrieval_utils import preprocess_localized_data, compute_retrieval_metrics

pp = pprint.PrettyPrinter(indent=4)
STORAGE_DIR = "/home/our_storage/"
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")


print("Task: Evaluate retrieval on COCO")

# load model and preprocesses
model_name = "blip2"
model_type = "pretrain"
cfg = OmegaConf.load(LocalizedBlip2QFormer.default_config_path(model_type)).preprocess
vis_preprocess, text_preprocess = load_preprocess(cfg)

vg_dataset = VGDataset(
    image_dir=os.path.join(STORAGE_DIR, "shared/datasets/vg/images"),
    index_file=os.path.join(STORAGE_DIR, "shared/datasets/vg/annotations/vg_bigger_than_median.json"),
    vis_processor=vis_preprocess,
    text_processor=text_preprocess,
)
vg_val_dataloader = DataLoader(
    vg_dataset.get_split("val"),
    batch_size=64,
    num_workers=4,
    collate_fn=VGDataset.collate_fn
)

2023-08-01 00:16:31.108730: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-08-01 00:16:31.163202: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-08-01 00:16:32.067516: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-08-01 00:16:33.301583: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at htt

Task: Evaluate retrieval on COCO
Loading from index file


In [3]:
# from collections import OrderedDict

model = LocalizedBlip2QFormer.from_pretrained(model_type=model_type).to("cuda")

# checkpoint = torch.load("/home/our_storage/shared/epoch_2.ckpt")

# # change key names
# new_state_dict = OrderedDict()
# for k, v in checkpoint["state_dict"].items():
#     name = k.replace("model.", "")
#     new_state_dict[name] = v

# checkpoint['state_dict'] = new_state_dict


In [4]:
model_cfg = OmegaConf.load("/home/our_storage/liam_sbhoo/dl-lab-final-project/configs/localized_qformer_model.yaml")
model = LocalizedBlip2QFormer.from_config(model_cfg.model)
print("loading checkpoint...")
model.load_state_dict(checkpoint["state_dict"])
model.to(device)

loading checkpoint...


LocalizedBlip2QFormer(
  (visual_encoder): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (blocks): ModuleList(
      (0-38): 39 x Block(
        (norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=1408, out_features=4224, bias=False)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1408, out_features=1408, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=1408, out_features=6144, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=6144, out_features=1408, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
     

In [4]:
all_images, all_texts, all_bbox_info, i2t, t2i = preprocess_localized_data(vg_val_dataloader, group_by="ann_id")

preprocess data
generate i2t and t2i ground truth


In [5]:
with torch.cuda.amp.autocast():
        score_matrix_i2t, score_matrix_t21 = model.compute_sim_matrix(all_images, all_texts, all_bbox_info, topk=128)

Computing features for evaluation...
Computing text embeddings...


Computing image embeddings...
Done generating embeddings


Evaluation: Computing similarity matrix: 100%|██████████| 5001/5001 [09:59<00:00,  8.34it/s]


In [6]:
score_matrix_i2t.shape, score_matrix_t21.shape

((5001, 5001), (5001, 5001))

In [7]:
len(i2t)

5001

In [8]:
retrieval_metrics = compute_retrieval_metrics(score_matrix_i2t, score_matrix_t21, t2i, i2t)
retrieval_metrics

{'txt_r1': 4.839032193561287,
 'txt_r5': 20.235952809438114,
 'txt_r10': 34.1131773645271,
 'txt_r_mean': 19.72938745584217,
 'img_r1': 5.658868226354729,
 'img_r5': 23.075384923015395,
 'img_r10': 36.75264947010598,
 'img_r_mean': 21.828967539825367,
 'r_mean': 20.779177497833768,
 'agg_metrics': 19.72938745584217}